# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, following the Croissant standard. All entities (record sets, fields, columns) are referenced by their `@id`.

### Dataset Source
The dataset is described by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # keep as object, do not subscript

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, their fields, and associated IDs. This helps you explore and decide what to analyze.

We'll print each record set `@id`, its name, and its fields/columns by `@id`.

In [ ]:
print("Available record sets and their fields:\n")
for record_set in metadata.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', '')}")
    print(f"  Fields/Columns:")
    
    # Print columns if available
    if hasattr(record_set, 'columns') and record_set.columns:
        for col in record_set.columns:
            print(f"    • Column @id: {col.id}, name: {getattr(col, 'name', '')}")
    # Or print fields
    if hasattr(record_set, 'fields') and record_set.fields:
        for field in record_set.fields:
            print(f"    • Field @id: {field.id}, name: {getattr(field, 'name', '')}")
    print("\n---\n")

## 3. Data Extraction
Now, let's load the main tabular record sets as pandas DataFrames. When available, use the `@id` of the record set for extraction.

In [ ]:
# You may need to adjust the record_set_ids below based on output above.

# Get all record_set @id(s)
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load up to 1000 records for preview
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for: {record_set_id}, {dataframes[record_set_id].shape[0]} rows, {dataframes[record_set_id].shape[1]} columns.")
    else:
        print(f"No records found for: {record_set_id} (maybe not a tabular or populated record set)")

# Preview columns and the first few records of the main data (replace <main_record_set_id> below if needed):
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in DataFrame '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No tabular dataframes were found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field/column for simple filtering, normalization, and grouping for demonstration.  
Replace the IDs below with those printed in the overview as needed.

In [ ]:
import numpy as np
# Choose main_record_set_id and numeric_field based on results above
if len(dataframes) > 0:
    # You may change these based on your actual fields:
    df = dataframes[main_record_set_id]
    # Try to auto-detect a numeric field
    candidate_numeric = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[col])]
    if len(candidate_numeric) == 0:
        # Try to convert columns to numeric where possible
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            num_na = coerced.isna().sum()
            if num_na < len(df) * 0.2:  # at least 80% values parsable
                df[col] = coerced
                candidate_numeric.append(col)
    if len(candidate_numeric):
        numeric_field = candidate_numeric[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.4f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try group by a categorical column (excluding the numeric field)
        group_candidates = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or pd.api.types.is_categorical_dtype(df[c]))]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nAverage {numeric_field} grouped by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No clear categorical group field found to group by.")
    else:
        print("No numeric field detected in main data set. Please adjust field selection above.")
else:
    print("No loaded DataFrames available for EDA.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and, if available, visualize the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if len(dataframes) > 0 and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Average {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Average {numeric_field}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print('Visualization skipped: no numeric field or dataframe.')

## 6. Conclusion
This notebook guided you through loading, exploring, and processing the FAIR^2 dataset on Human Mast Cell EV protein abundances with `mlcroissant`.

- **Metadata and record set IDs** are discoverable programmatically.
- You can select and manipulate fields using their Croissant `@id` or names, depending on your extraction needs.
- Standard EDA and visualization workflows are straightforward once the tabular data is accessed.

[Learn more about Croissant standard and FAIR^2 on senscience.ai](https://sen.science/doi/10.71728/senscience.vq4a-28xa)